# TCC WikiArt — Etapa 9: Avaliação Quantitativa

## 1. Ambiente e dependências


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /content

!rm -rf tcc_wikiart
!git clone https://github.com/fbrum44/tcc_wikiart.git

%cd /content/tcc_wikiart

In [ ]:
!git log -1 --oneline
!git status --short

In [ ]:
!pip install -q datasets opencv-python scikit-learn timm

In [ ]:
import sklearn
import timm
import pandas as pd

print("scikit-learn:", sklearn.__version__)
print("timm:", timm.__version__)

In [ ]:
%env HF_HOME=/content/hf_cache

## 2. Dataset WikiArt

In [ ]:
from datasets import load_dataset

wikiart = load_dataset(
    "huggan/wikiart",
    split="train",
    keep_in_memory=False,
)

print("Imagens WikiArt:", len(wikiart))

if len(wikiart) != 81444:
    raise RuntimeError(
        f"Esperadas 81.444 imagens, encontradas {len(wikiart)}."
    )

## 3. Google Drive e checkpoints

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/TCC_WikiArt_Compartilhado"
)

MODELS_DRIVE = DRIVE_ROOT / "models"

print("Pasta compartilhada:", DRIVE_ROOT.exists())
print("Models:", MODELS_DRIVE.exists())

In [ ]:
checkpoints = [
    MODELS_DRIVE / "resnet50_finetune_best.pt",
    MODELS_DRIVE / "vit_finetune_best.pt",
]

for checkpoint in checkpoints:
    print(
        checkpoint.name,
        "→",
        "OK" if checkpoint.exists() else "NÃO ENCONTRADO"
    )

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("/content/tcc_wikiart")
LOCAL_MODELS = REPO_DIR / "models"

if LOCAL_MODELS.is_symlink():
    LOCAL_MODELS.unlink()
elif LOCAL_MODELS.exists():
    shutil.rmtree(LOCAL_MODELS)

os.symlink(
    MODELS_DRIVE,
    LOCAL_MODELS,
    target_is_directory=True,
)

print("models ->", LOCAL_MODELS.resolve())

In [ ]:
!ls -lh models/

## 4. Smoke tests na validação

In [ ]:
!python -m src.evaluation.evaluate_model \
    --model resnet50 \
    --checkpoint models/resnet50_finetune_best.pt \
    --split validation \
    --batch-size 32 \
    --num-workers 2 \
    --max-batches 3

In [ ]:
!find outputs/evaluation/resnet50 -maxdepth 1 -type f -printf "%f\n"

In [ ]:
import json
from pathlib import Path

path = Path(
    "outputs/evaluation/resnet50/"
    "validation_smoke_patch_summary.json"
)

with path.open(
    encoding="utf-8"
) as f:
    resumo = json.load(f)

resumo

In [ ]:
import pandas as pd

df_smoke = pd.read_csv(
    "outputs/evaluation/resnet50/"
    "validation_smoke_patch_predictions.csv.gz"
)

print(df_smoke["true_class"].value_counts())
print()
print(df_smoke["pred_class"].value_counts())

In [ ]:
!python -m src.evaluation.evaluate_model \
    --model vit_b16 \
    --checkpoint models/vit_finetune_best.pt \
    --split validation \
    --batch-size 32 \
    --num-workers 2 \
    --max-batches 3

## 5. Avaliação oficial da ResNet-50

In [ ]:
!python -m src.evaluation.evaluate_model \
    --model resnet50 \
    --checkpoint models/resnet50_finetune_best.pt \
    --split test \
    --batch-size 32 \
    --num-workers 2

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/tcc_wikiart")
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/TCC_WikiArt_Compartilhado"
)

ORIGEM = (
    REPO_DIR
    / "outputs"
    / "evaluation"
    / "resnet50"
)

DESTINO = (
    DRIVE_ROOT
    / "outputs"
    / "evaluation"
    / "resnet50"
)

DESTINO.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in ORIGEM.iterdir():
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            DESTINO / arquivo.name,
        )
        print("Backup:", arquivo.name)

print("\nBackup ResNet concluído.")

## 6. Avaliação oficial do ViT-B/16


In [ ]:
!python -m src.evaluation.evaluate_model \
    --model vit_b16 \
    --checkpoint models/vit_finetune_best.pt \
    --split test \
    --batch-size 32 \
    --num-workers 2

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/tcc_wikiart")
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/TCC_WikiArt_Compartilhado"
)

ORIGEM = (
    REPO_DIR
    / "outputs"
    / "evaluation"
    / "vit_b16"
)

DESTINO = (
    DRIVE_ROOT
    / "outputs"
    / "evaluation"
    / "vit_b16"
)

DESTINO.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in ORIGEM.iterdir():
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            DESTINO / arquivo.name,
        )
        print("Backup:", arquivo.name)

print("\nBackup ViT concluído.")

## 7. Agregação das previsões por pintura


In [ ]:
import pandas as pd
from pathlib import Path

arquivos = {
    "ResNet": Path(
        "outputs/evaluation/resnet50/"
        "test_patch_predictions.csv.gz"
    ),
    "ViT": Path(
        "outputs/evaluation/vit_b16/"
        "test_patch_predictions.csv.gz"
    ),
}

for nome, caminho in arquivos.items():
    df = pd.read_csv(caminho)

    print(f"\n{nome}")
    print("Patches:", len(df))
    print(
        "Pinturas:",
        df["Painting_ID"].nunique()
    )
    print(
        "Classes:",
        sorted(
            df["true_class"].unique()
        )
    )

    inconsistencias = (
        df.groupby("Painting_ID")["true_idx"]
        .nunique()
        .gt(1)
        .sum()
    )

    print(
        "Pinturas com mais de um "
        "rótulo real:",
        inconsistencias,
    )

In [ ]:
!git pull --rebase origin main

In [ ]:
!git status --short

## 8. Auditoria das agregações


In [ ]:
!python -m src.evaluation.aggregate_painting \
    --model resnet50 \
    --predictions outputs/evaluation/resnet50/test_patch_predictions.csv.gz

In [ ]:
!python -m src.evaluation.aggregate_painting \
    --model vit_b16 \
    --predictions outputs/evaluation/vit_b16/test_patch_predictions.csv.gz

## 9. Comparação final ResNet × ViT


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CLASSES = [
    "Renaissance",
    "Baroque",
    "Impressionism",
    "Cubism",
]

for modelo in ["resnet50", "vit_b16"]:

    caminho = Path(
        f"outputs/evaluation/{modelo}/"
        "test_painting_predictions.csv.gz"
    )

    df = pd.read_csv(caminho)

    print("\n" + "=" * 60)
    print(modelo)
    print("=" * 60)

    # 1. Recalcular predição pela média
    prob_cols = [
        f"mean_prob_{classe}"
        for classe in CLASSES
    ]

    pred_mean_recalculado = (
        df[prob_cols]
        .to_numpy()
        .argmax(axis=1)
    )

    print(
        "Predição média reproduzível:",
        np.array_equal(
            pred_mean_recalculado,
            df["pred_mean_idx"].to_numpy()
        )
    )

    # 2. Recalcular maioria onde não houve empate
    vote_cols = [
        f"votes_{classe}"
        for classe in CLASSES
    ]

    votos = df[vote_cols].to_numpy()

    maior = votos.max(axis=1)
    n_empates = (
        (votos == maior[:, None])
        .sum(axis=1)
    )

    sem_empate = n_empates == 1

    maioria_recalculada = (
        votos.argmax(axis=1)
    )

    print(
        "Maioria reproduzível sem empates:",
        np.array_equal(
            maioria_recalculada[sem_empate],
            df.loc[
                sem_empate,
                "pred_majority_idx"
            ].to_numpy()
        )
    )

    print(
        "Empates:",
        (~sem_empate).sum()
    )

    # 3. O rótulo real não é igual artificialmente
    print(
        "Erros pela média:",
        (
            df["true_idx"]
            != df["pred_mean_idx"]
        ).sum()
    )

    print(
        "Erros pela maioria:",
        (
            df["true_idx"]
            != df["pred_majority_idx"]
        ).sum()
    )

In [ ]:
!git pull --rebase origin main

In [ ]:
!python -m src.evaluation.compare_models

In [ ]:
import pandas as pd

classes = pd.read_csv(
    "outputs/evaluation/comparison_by_class.csv"
)

display(
    classes[
        (
            classes["level"] == "painting"
        )
        & (
            classes["aggregation"]
            == "mean_probability"
        )
    ]
)

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/tcc_wikiart")

ORIGEM = (
    REPO_DIR
    / "outputs"
    / "evaluation"
)

DESTINO = (
    Path(
        "/content/drive/MyDrive/"
        "TCC_WikiArt_Compartilhado/"
        "outputs/evaluation"
    )
)

DESTINO.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in ORIGEM.rglob("*"):

    if not arquivo.is_file():
        continue

    relativo = arquivo.relative_to(
        ORIGEM
    )

    destino = (
        DESTINO
        / relativo
    )

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        arquivo,
        destino,
    )

print(
    "Backup completo da Etapa 9 concluído."
)

## 10. Matrizes de confusão


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

CLASSES = [
    "Renaissance",
    "Baroque",
    "Impressionism",
    "Cubism",
]

for modelo in ["resnet50", "vit_b16"]:

    caminho = (
        f"outputs/evaluation/{modelo}/"
        "test_painting_predictions.csv.gz"
    )

    df = pd.read_csv(caminho)

    matriz = confusion_matrix(
        df["true_idx"],
        df["pred_mean_idx"],
        labels=[0, 1, 2, 3],
    )

    matriz_df = pd.DataFrame(
        matriz,
        index=[f"Real: {c}" for c in CLASSES],
        columns=[f"Pred: {c}" for c in CLASSES],
    )

    print("\n" + "=" * 70)
    print(modelo)
    print("=" * 70)

    display(matriz_df)

## 11. Backup dos resultados

In [ ]:
for modelo in ["resnet50", "vit_b16"]:

    caminho = (
        f"outputs/evaluation/{modelo}/"
        "test_painting_predictions.csv.gz"
    )

    df = pd.read_csv(caminho)

    matriz = confusion_matrix(
        df["true_idx"],
        df["pred_mean_idx"],
        labels=[0, 1, 2, 3],
        normalize="true",
    )

    matriz_df = pd.DataFrame(
        matriz * 100,
        index=[f"Real: {c}" for c in CLASSES],
        columns=[f"Pred: {c}" for c in CLASSES],
    )

    print("\n" + "=" * 70)
    print(modelo, "— percentuais")
    print("=" * 70)

    display(
        matriz_df.round(2)
    )